# Module 33 — Exercise 2: Cache Stampede Mitigation via Single-Flight Locking

When a hot cache key expires under 1,000 requests/sec, all 1,000 requests concurrently miss and query the database (the thundering herd / stampede problem).

In this exercise, you will implement Single-Flight (request coalescing) to ensure only ONE worker recomputes the key while all other callers wait on the same result.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 33 README, Module 22 |



# Your turn


### Task 1: Async Single-Flight Coordinator

Implement `SingleFlightGroup` with method `async do(key, async_fn)`:
- If a computation for `key` is already in flight, return the existing Task.
- Otherwise, create and launch the Task, ensuring it is cleaned up once completed.


In [ ]:
# ANSWER 1
import asyncio

class SingleFlightGroup:
    def __init__(self):
        self._in_flight: dict[str, asyncio.Task] = {}

    async def do(self, key: str, fn, *args, **kwargs):
        if key in self._in_flight:
            return await self._in_flight[key]

        task = asyncio.create_task(fn(*args, **kwargs))
        self._in_flight[key] = task
        try:
            return await task
        finally:
            self._in_flight.pop(key, None)



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

async def main_check():
    group = SingleFlightGroup()
    db_calls = 0

    async def expensive_query(key):
        nonlocal db_calls
        db_calls += 1
        await asyncio.sleep(0.05)
        return f"result_for_{key}"

    # 10 concurrent requests for the exact same key
    results_list = await asyncio.gather(*(group.do("hot_key", expensive_query, "hot_key") for _ in range(10)))
    
    results = [
        check(db_calls == 1, "Task 1: Exactly 1 database query executed despite 10 concurrent requests"),
        check(all(r == "result_for_hot_key" for r in results_list), "Task 1: All 10 callers received correct result"),
    ]
    print(f"Summary: {sum(results)}/{len(results)} checks passed.")

await main_check()

